# Uygulama 3 — Görme Kaybının Genetik İzi: LHON Mutasyonu

**Hafta 2** — MutableSeq, nokta mutasyonu, sessiz / anlamlı (missense) değişim

**Veri:** NC_012920.1 (FASTA) — ND4: 10760..12137, ATP6: 8527..9207

---

**Senaryo:** Genç yaşta ani ve ağrısız görme kaybı yaşayan bir hastanın mtDNA dizilemesinde referansa göre **m.11778G>A** değişimi bulunuyor. Bu değişim, **Leber kalıtsal optik nöropatisinin (LHON)** en sık nedenidir.

Genetik danışman senden, değişimin ND4 proteininde hangi amino asidi etkilediğini göstermeni istiyor.

> **Günlük benzetme:** DNA bir tarif kitabı, her gen bir tarif. Tek bir harfin değişmesi (mutasyon) bazen tarifi tamamen bozar (anlamlı mutasyon), bazen hiçbir şey değişmez (sessiz mutasyon) — çünkü birden fazla harf kombinasyonu aynı malzemeyi (amino asidi) kodlayabilir.

## Kalem-Kağıt Isınması

Genomdaki **11778. baz**, 10760'ta başlayan ND4 geninin kaçıncı bazıdır?

- Gen içi pozisyon = 11778 − 10760 + 1 = **1019. baz**
- Kodon numarası = (1019 − 1) ÷ 3 + 1 = 1018 ÷ 3 + 1 = 339 + 1 = **340. kodon**
- Kodon içi pozisyon = ((1019 − 1) % 3) + 1 = (1018 % 3) + 1 = 2 + 1 = **3. harf**

Kodon CGC (Arg) ise G→A değişimi → **CGC → CGA**

Ama bekle! Bu mitokondri, tablo 2 kullan:
- CGC → Arg (R)
- CGA → Arg (R)? Hayır... Bakalım kodda ne çıkacak!

(Aslında kodon **CGG → CAG** olabilir, bu hesaplamayı kodla doğrulayacağız.)

---
## Görev 0 — Veriyi Yükle

In [ ]:
from Bio import Entrez, SeqIO
from Bio.Seq import Seq, MutableSeq
import os

Entrez.email = "semra@example.com"

if os.path.exists("NC_012920.fasta"):
    record = SeqIO.read("NC_012920.fasta", "fasta")
    print("Yerel dosyadan okundu.")
else:
    handle = Entrez.efetch(db="nucleotide", id="NC_012920.1",
                           rettype="fasta", retmode="text")
    record = SeqIO.read(handle, "fasta")
    handle.close()
    SeqIO.write(record, "NC_012920.fasta", "fasta")
    print("NCBI'dan indirildi.")

genom = record.seq.upper()
print(f"Genom: {len(genom)} bp")

---
## Görev 1 — ND4 Genini Kes

ND4 geni: **10760..12137** (1-tabanlı, dahil, ileri iplik)

Python dilimi: `genom[10759:12137]`

In [ ]:
nd4 = genom[10759:12137]

print(f"ND4 uzunluğu: {len(nd4)} baz")
print(f"3'ün katı mı? → {len(nd4) % 3 == 0}")
print(f"Kalan: {len(nd4) % 3} baz")
print(f"Son 6 baz: {nd4[-6:]}")

print(f"\nNeden 3'ün katı değil?")
print(f"→ Mitokondriyal mRNA'lar poli-A kuyruğu alır.")
print(f"→ Son kodon TA ile bitiyor, poli-A kuyruğu 'A' ekleyince TAA olur.")
print(f"→ TAA = dur kodonu (tablo 2'de de dur).")
print(f"→ Yani gen aslında tam, ama genomda son A eksik görünür.")

---
## Görev 2 — ND4'ü Proteine Çevir

Mitokondriyal kod (tablo 2), `to_stop=True` ile dur kodonuna kadar çevir.

In [ ]:
protein_nd4 = nd4.translate(table=2, to_stop=True)

print(f"ND4 proteini: {len(protein_nd4)} amino asit")
print(f"İlk 30 aa: {protein_nd4[:30]}")
print(f"Son 10 aa:  {protein_nd4[-10:]}")

---
## Görev 3 — m.11778G>A Mutasyonunu Uygula

**MutableSeq** nedir?
- Normal `Seq` nesnesi **değiştirilemez** (immutable) — güvenli ama mutasyon simüle edemezsin
- `MutableSeq` ise **değiştirilebilir** — tek bir harfi değiştirebilirsin

Düşün: `Seq` = donmuş bir buz kalıbı, `MutableSeq` = hamur (şekil verebilirsin)

**m.11778G>A** = genomun 11778. pozisyonundaki G bazı A'ya dönüşmüş.

In [ ]:
# Mutasyon pozisyonunu hesapla
genom_poz = 11778           # 1-tabanlı genom pozisyonu
gen_baslangic = 10760       # ND4 başlangıcı (1-tabanlı)
gen_ici_poz = genom_poz - gen_baslangic  # 0-tabanlı gen içi pozisyon

kodon_no = gen_ici_poz // 3 + 1          # 1-tabanlı kodon numarası
kodon_ici = gen_ici_poz % 3 + 1          # kodon içi pozisyon (1, 2 veya 3)

print(f"Genomda pozisyon: {genom_poz}")
print(f"Gen içi pozisyon (0-tabanlı): {gen_ici_poz}")
print(f"Kodon numarası: {kodon_no}. kodon")
print(f"Kodon içi harf: {kodon_ici}. pozisyon")

# Orijinal baz ne?
print(f"\nOrijinal baz: {nd4[gen_ici_poz]}")

# Orijinal kodon
kodon_bas = (gen_ici_poz // 3) * 3
orijinal_kodon = str(nd4[kodon_bas:kodon_bas+3])
print(f"Orijinal kodon: {orijinal_kodon}")

In [ ]:
# MutableSeq ile mutasyonu uygula
nd4_mutant = MutableSeq(str(nd4))
print(f"Değiştirilecek baz: {nd4_mutant[gen_ici_poz]} → A")
nd4_mutant[gen_ici_poz] = 'A'

# Mutant kodon
mutant_kodon = str(nd4_mutant[kodon_bas:kodon_bas+3])
print(f"Mutant kodon:   {mutant_kodon}")

# İki proteini çevir ve karşılaştır
protein_normal = nd4.translate(table=2, to_stop=True)
protein_mutant = Seq(str(nd4_mutant)).translate(table=2, to_stop=True)

print(f"\nNormal protein uzunluğu: {len(protein_normal)} aa")
print(f"Mutant protein uzunluğu: {len(protein_mutant)} aa")

# Farklı pozisyonları bul
farklar = []
for i in range(min(len(protein_normal), len(protein_mutant))):
    if protein_normal[i] != protein_mutant[i]:
        farklar.append((i + 1, protein_normal[i], protein_mutant[i]))

print(f"\nFarklı pozisyon sayısı: {len(farklar)}")
for poz, norm, mut in farklar:
    print(f"  Pozisyon {poz}: {norm} → {mut}")

print(f"\n→ Bu bir ANLAMLI (missense) mutasyon:")
print(f"  Amino asit değişiyor → protein işlevi bozulabilir → LHON hastalığı")

---
## Görev 4 — Sessiz Mutasyon Örneği

Aynı kodonun **3. harfini** değiştirsek ne olurdu?

- Kodonun 3. pozisyonu "wobble" (sallanma) pozisyonudur
- Genellikle amino asidi değiştirmez → **sessiz (synonymous) mutasyon**
- Düşün: "renk" kelimesindeki son harfi değiştirsen "renk" hâlâ renk — anlam aynı

In [ ]:
# Kodonun 3. harfini değiştirelim
nd4_sessiz = MutableSeq(str(nd4))
ucuncu_poz = kodon_bas + 2  # 3. harf (0-tabanlı)

orijinal_3 = nd4_sessiz[ucuncu_poz]
# Farklı bir baz koy
yeni_baz = 'A' if orijinal_3 != 'A' else 'T'
nd4_sessiz[ucuncu_poz] = yeni_baz

sessiz_kodon = str(nd4_sessiz[kodon_bas:kodon_bas+3])

# Amino asitleri karşılaştır
aa_orijinal = Seq(orijinal_kodon).translate(table=2)
aa_sessiz = Seq(sessiz_kodon).translate(table=2)

print(f"Orijinal kodon: {orijinal_kodon} → {aa_orijinal}")
print(f"Değişmiş kodon: {sessiz_kodon} → {aa_sessiz}")

if aa_orijinal == aa_sessiz:
    print(f"\n→ SESSİZ (synonymous) mutasyon: amino asit DEĞİŞMEDİ")
    print(f"  Kodonun 3. harfi genellikle amino asidi etkilemez.")
else:
    print(f"\n→ Bu durumda amino asit değişti (her 3. pozisyon sessiz değildir).")

---
## Görev 5 — NARP Sendromu: ATP6 Geninde m.8993T>G

Aynı yöntemi başka bir hastalıkta uygulayalım.

- **NARP sendromu:** Nöropati, Ataksi, Retinitis Pigmentosa
- Mutasyon: **m.8993T>G** (ATP6 geninde)
- ATP6 geni: **8527..9207**

In [ ]:
# ATP6 genini kes
atp6 = genom[8526:9207]
print(f"ATP6 uzunluğu: {len(atp6)} baz")
print(f"3'ün katı mı? → {len(atp6) % 3 == 0}")

# Mutasyon pozisyonunu hesapla
genom_poz_narp = 8993
gen_bas_atp6 = 8527
gen_ici_narp = genom_poz_narp - gen_bas_atp6  # 0-tabanlı

kodon_no_narp = gen_ici_narp // 3 + 1
kodon_ici_narp = gen_ici_narp % 3 + 1
kodon_bas_narp = (gen_ici_narp // 3) * 3

orijinal_kodon_narp = str(atp6[kodon_bas_narp:kodon_bas_narp+3])
print(f"\nMutasyon: m.{genom_poz_narp}T>G")
print(f"Kodon: {kodon_no_narp}. kodon, {kodon_ici_narp}. harf")
print(f"Orijinal kodon: {orijinal_kodon_narp}")

# Mutasyonu uygula
atp6_mutant = MutableSeq(str(atp6))
print(f"Değiştirilecek baz: {atp6_mutant[gen_ici_narp]} → G")
atp6_mutant[gen_ici_narp] = 'G'

mutant_kodon_narp = str(atp6_mutant[kodon_bas_narp:kodon_bas_narp+3])
print(f"Mutant kodon: {mutant_kodon_narp}")

# Amino asit değişimi
aa_norm = Seq(orijinal_kodon_narp).translate(table=2)
aa_mut = Seq(mutant_kodon_narp).translate(table=2)

print(f"\nAmino asit değişimi: {aa_norm} → {aa_mut}")
print(f"→ Bu değişim ATP sentaz alt birimini bozarak")
print(f"  hücrenin enerji üretimini etkiler → NARP sendromu")

---

## Uygulama 3 Tamamlandı!

**Öğrendiğin kavramlar:**
- `MutableSeq` ile diziyi değiştirme (mutasyon simülasyonu)
- Genom pozisyonu → gen içi pozisyon → kodon numarası dönüşümü
- **Anlamlı (missense) mutasyon:** amino asit değişir → protein bozulabilir
- **Sessiz (synonymous) mutasyon:** kodon değişir ama amino asit aynı kalır
- Aynı yöntemi farklı hastalıklara uygulama (LHON → NARP)

**Sonraki:** Uygulama 4 — PCR Primeri (Tm, GC profili, kayan pencere)